# Silver Layer — Base auxiliar
Reads raw boletos from the **Bronze** Delta table, applies cleaning and enrichment transformations, and writes the result to the **Silver** Delta table.

**Medallion flow:** `Bronze (raw CSV)` → **`Silver (cleaned & enriched)`** → Gold (aggregated)

## 1. Imports & SparkSession

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, TimestampType, DateType, StructType, StructField, StringType, BooleanType, IntegerType
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName('Silver Base auxiliar')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print(f'Spark version: {spark.version}')

Spark version: 4.1.1


## 2. Read from Bronze
Load the raw Delta table written by the ingestion notebook.

In [3]:
BRONZE_PATH = '../../output_data/bronze/base_auxiliar'

schema = StructType([
    StructField('id_cnpj',       StringType(), nullable=True),
    StructField('cd_cnae_prin',      StringType(), nullable=True),
    StructField('uf', StringType(), nullable=True),
    StructField('sacado_indice_liquidez_1m',      StringType(), nullable=True),
    StructField('cedente_indice_liquidez_1m',   StringType(), nullable=True),
    StructField('score_materialidade_evolucao',    StringType(), nullable=True),
    StructField('media_atraso_dias',     DoubleType(), nullable=True),
    StructField('indicador_liquidez_quantitativo_3m',       DoubleType(), nullable=True),
    StructField('share_vl_inad_pag_bol_6_a_15d',      StringType(), nullable=True),
    StructField('score_quantidade_v2',    StringType(), nullable=True),
    StructField('score_materialidade_v2',    StringType(), nullable=True),

    StructField('partition_date',   DateType(), nullable=False),
    StructField('ingestion_timestamp', TimestampType(), nullable=False)
])

df_bronze = spark.read.format('delta').load(BRONZE_PATH)

print(f'Rows in bronze: {df_bronze.count():,}')
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

Rows in bronze: 4,612
root
 |-- id_cnpj: string (nullable = true)
 |-- cd_cnae_prin: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sacado_indice_liquidez_1m: string (nullable = true)
 |-- cedente_indice_liquidez_1m: string (nullable = true)
 |-- score_materialidade_evolucao: string (nullable = true)
 |-- media_atraso_dias: double (nullable = true)
 |-- indicador_liquidez_quantitativo_3m: double (nullable = true)
 |-- share_vl_inad_pag_bol_6_a_15d: string (nullable = true)
 |-- score_quantidade_v2: string (nullable = true)
 |-- score_materialidade_v2: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)

+----------------------------------------------------------------+------------+---+-------------------------+--------------------------+----------------------------+-----------------+----------------------------------+-----------------------------+-------------------+----------------------+----------

## 3. Data Quality Checks
Before transforming, understand what needs to be fixed.

In [4]:
total = df_bronze.count()

null_counts = df_bronze.select([
    F.sum(F.col(c).isNull().cast('int')).alias(c)
    for c in df_bronze.columns
])

null_pct = null_counts.select([
    F.round(F.col(c) / total * 100, 2).alias(c)
    for c in df_bronze.columns
])

print('Null counts:')
null_counts.show(truncate=False)
print('Null % per column:')
null_pct.show(truncate=False)

Null counts:
+-------+------------+---+-------------------------+--------------------------+----------------------------+-----------------+----------------------------------+-----------------------------+-------------------+----------------------+-------------------+--------------+
|id_cnpj|cd_cnae_prin|uf |sacado_indice_liquidez_1m|cedente_indice_liquidez_1m|score_materialidade_evolucao|media_atraso_dias|indicador_liquidez_quantitativo_3m|share_vl_inad_pag_bol_6_a_15d|score_quantidade_v2|score_materialidade_v2|ingestion_timestamp|partition_date|
+-------+------------+---+-------------------------+--------------------------+----------------------------+-----------------+----------------------------------+-----------------------------+-------------------+----------------------+-------------------+--------------+
|0      |2           |359|19                       |2149                      |3                           |5                |20                                |5               

In [5]:
total_rows   = df_bronze.count()
unique_cnpjs   = df_bronze.select('id_cnpj').distinct().count()
duplicates   = total_rows - unique_cnpjs

print(f'Total rows    : {total_rows:,}')
print(f'Unique cnpjs: {unique_cnpjs:,}')
print(f'Duplicates    : {duplicates:,}')

Total rows    : 4,612
Unique cnpjs: 4,612
Duplicates    : 0


## 4. Transformations
Apply all cleaning and enrichment steps to produce the silver DataFrame.

In [6]:

df_silver = (
    df_bronze
    .withColumn("grupo_cnae",
        F.substring(F.col("cd_cnae_prin").cast("string"), 1, 2))
    .withColumn("setor_economico",
        F.when(F.col("grupo_cnae").isin("01","02","03"), "Agropecuaria")
         .when(F.col("grupo_cnae").between("05", "09"), "Extrativa")
         .when(F.col("grupo_cnae").between("10", "33"), "Industria")
         .when(F.col("grupo_cnae").between("35", "39"), "Utilidades")
         .when(F.col("grupo_cnae").between("41", "43"), "Construcao")
         .when(F.col("grupo_cnae").between("45", "47"), "Comercio")
         .when(F.col("grupo_cnae").between("49", "53"), "Transporte")
         .when(F.col("grupo_cnae").between("55", "56"), "Alojamento/Alimentacao")
         .when(F.col("grupo_cnae").between("58", "63"), "Informacao/Comunicacao")
         .when(F.col("grupo_cnae").between("64", "66"), "Financeiro")
         .otherwise("Servicos/Outros"))

    # --- Região do Brasil ---
    .withColumn("regiao",
        F.when(F.col("uf").isin("SP","RJ","MG","ES"), "Sudeste")
         .when(F.col("uf").isin("PR","SC","RS"), "Sul")
         .when(F.col("uf").isin("BA","PE","CE","MA","PI","RN","PB","SE","AL"), "Nordeste")
         .when(F.col("uf").isin("AM","PA","AP","RO","RR","AC","TO"), "Norte")
         .when(F.col("uf").isin("GO","DF","MT","MS"), "Centro-Oeste")
         .otherwise("Nao Informado"))

    # --- Faixa de liquidez ---
    .withColumn("faixa_liquidez_sacado",
        F.when(F.col("sacado_indice_liquidez_1m") < 0.3, "Baixa")
         .when(F.col("sacado_indice_liquidez_1m") < 0.6, "Media")
         .when(F.col("sacado_indice_liquidez_1m") < 0.85, "Alta")
         .otherwise("Muito Alta"))

    # --- Faixa de score ---
    .withColumn("faixa_score_materialidade",
        F.when(F.col("score_materialidade_v2") < 300, "Muito Baixo")
         .when(F.col("score_materialidade_v2") < 600, "Baixo")
         .when(F.col("score_materialidade_v2") < 800, "Medio")
         .when(F.col("score_materialidade_v2") < 950, "Alto")
         .otherwise("Muito Alto"))

    # --- Flag inadimplencia significativa ---
    .withColumn("flag_inadimplencia_alta",
        F.when(F.col("share_vl_inad_pag_bol_6_a_15d") > 0.15, True).otherwise(False))


    .withColumn('ingestion_timestamp', F.current_timestamp())

    .withColumn('partition_date', F.current_date())
)

df_silver.printSchema()

root
 |-- id_cnpj: string (nullable = true)
 |-- cd_cnae_prin: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sacado_indice_liquidez_1m: string (nullable = true)
 |-- cedente_indice_liquidez_1m: string (nullable = true)
 |-- score_materialidade_evolucao: string (nullable = true)
 |-- media_atraso_dias: double (nullable = true)
 |-- indicador_liquidez_quantitativo_3m: double (nullable = true)
 |-- share_vl_inad_pag_bol_6_a_15d: string (nullable = true)
 |-- score_quantidade_v2: string (nullable = true)
 |-- score_materialidade_v2: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = false)
 |-- partition_date: date (nullable = false)
 |-- grupo_cnae: string (nullable = true)
 |-- setor_economico: string (nullable = false)
 |-- regiao: string (nullable = false)
 |-- faixa_liquidez_sacado: string (nullable = false)
 |-- faixa_score_materialidade: string (nullable = false)
 |-- flag_inadimplencia_alta: boolean (nullable = false)



## 5. Validation
Quick sanity checks before writing.

In [7]:
cnpj_not_null     = df_silver.filter(F.col('id_cnpj').isNotNull()).count()
cnpj_null     = df_silver.filter(F.col('id_cnpj').isNull()).count()
cnae     = df_silver.filter(F.col('cd_cnae_prin').isNull()).count()
cnae_not_null     = df_silver.filter(F.col('cd_cnae_prin').isNotNull()).count()


print(f'Total rows      : {df_silver.count():,}')
print(f'Cnpj not nulls            : {cnpj_not_null:,}')
print(f'Cnpj nulls            : {cnpj_null:,}')
print(f'Cnae nulls            : {cnae:,}')
print(f'Cnae not nulls            : {cnae_not_null:,}')


Total rows      : 4,612
Cnpj not nulls            : 4,612
Cnpj nulls            : 0
Cnae nulls            : 2
Cnae not nulls            : 4,610


## 6. Write to Silver
Persist the cleaned and enriched DataFrame as a Delta table.

In [8]:
SILVER_PATH = '../../output_data/silver/base_auxiliar'

(
    df_silver
    .write
    .format('delta')
    .mode('overwrite')
    .partitionBy('partition_date')
    .option('overwriteSchema', 'true')
    .save(SILVER_PATH)
)

print(f'Silver layer saved to {SILVER_PATH}')

Silver layer saved to ../../output_data/silver/base_auxiliar


In [9]:
df_check = spark.read.format('delta').load(SILVER_PATH)
print(f'Rows in silver: {df_check.count():,}')
df_check.printSchema()

Rows in silver: 4,612
root
 |-- id_cnpj: string (nullable = true)
 |-- cd_cnae_prin: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- sacado_indice_liquidez_1m: string (nullable = true)
 |-- cedente_indice_liquidez_1m: string (nullable = true)
 |-- score_materialidade_evolucao: string (nullable = true)
 |-- media_atraso_dias: double (nullable = true)
 |-- indicador_liquidez_quantitativo_3m: double (nullable = true)
 |-- share_vl_inad_pag_bol_6_a_15d: string (nullable = true)
 |-- score_quantidade_v2: string (nullable = true)
 |-- score_materialidade_v2: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- partition_date: date (nullable = true)
 |-- grupo_cnae: string (nullable = true)
 |-- setor_economico: string (nullable = true)
 |-- regiao: string (nullable = true)
 |-- faixa_liquidez_sacado: string (nullable = true)
 |-- faixa_score_materialidade: string (nullable = true)
 |-- flag_inadimplencia_alta: boolean (nullable = true)

